# Multi-dataset evaluation (Milestone 8)

Zero-shot evaluation of the Kvasir-trained checkpoints on Kvasir-SEG **test**
plus the external datasets (CVC-ClinicDB, CVC-ColonDB, ETIS, CVC-300), then one
config x dataset table.

Run the cells top to bottom. The whole notebook is safe to rerun:

- `scripts/prepare_datasets.py` is idempotent: an existing `data.npz` at the same
  size is skipped, so re-preparing costs nothing.
- `processed/` is copied to `/content` so evaluation reads locally; `raw/` and
  `processed/` on Drive stay the source of truth.
- `src.evaluate` writes one row per `(run, split, dataset, tta)`; `FORCE = True`
  replaces that row in place on a rerun instead of duplicating it.
- Checkpoints (`best.pt`, `last.pt`) and the results files live on Drive and
  survive a disconnect.

Checkpoints are selected by **Kvasir val Dice only** (`best.pt`). The external
datasets are never used to select or tune a checkpoint, and every number here is
single-seed and provisional, not a final result.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/awdtyo/Polyp-Segmentation.git"
REF = "main"                    # commit hash, tag or branch (a commit is most reproducible)

# Data tree. Raw datasets are unpacked on Drive at <DRIVE_DATA_ROOT>/raw/<folder>
# (folder names come from configs/datasets.yaml). processed/ is built there and
# then copied to /content so evaluation reads locally.
DRIVE_DATA_ROOT = "/content/drive/MyDrive/polyp-data"
LOCAL_DATA_ROOT = "/content/polyp-data"

# Runs (checkpoints + the results CSV) live on Drive.
RUNS_DIR = "/content/drive/MyDrive/polyp-runs"
RESULTS_CSV = f"{RUNS_DIR}/results.csv"

# None = every run folder under RUNS_DIR that has best.pt and config_resolved.yaml
# (smoke runs are skipped). Otherwise set a list of run folder names.
ONLY_RUNS = None
EXCLUDE_SUBSTR = ("_smoke",)

# Must match the runs' data.image_size; rebuild with --force to change it.
PREP_SIZE = 256
RUN_PREP = True                 # False: reuse an existing processed/ on Drive

EVAL_SPLIT = "test"             # 'val' reproduces the training log; 'test' is the held-out number
FORCE = True                    # replace a run/dataset row in place on a rerun
TTA_FLIP = False                # optional flip test-time augmentation, off by default

REPO_DIR = Path("/content/polyp")

## 1. Mount Drive, get the repo, check the GPU

`GH_TOKEN` is read from Colab Secrets (key icon in the left sidebar, scoped to
this notebook) and is never printed. The repo is fetched and checked out at
`REF` (a commit, tag or branch) instead of a plain `git pull`, so a rerun cannot
silently pick up newer code.

In [ ]:
import subprocess

from google.colab import drive, userdata

drive.mount("/content/drive")

token = userdata.get("GH_TOKEN")
if not token:
    raise RuntimeError(
        "Missing GH_TOKEN. Add it in Colab Secrets (key icon) and enable it for this notebook."
    )


def _auth_url(url: str, secret: str) -> str:
    """Insert a token into an https clone URL without storing it anywhere."""
    if not url.startswith("https://"):
        raise ValueError("REPO_URL must be an https URL")
    return url.replace("https://", f"https://x-access-token:{secret}@", 1)


def _git(args: list[str], check: bool = True) -> subprocess.CompletedProcess:
    """Run git and raise a token-free message on failure (unless check=False)."""
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if check and result.returncode != 0:
        message = (result.stderr or result.stdout).replace(token, "***")
        raise RuntimeError(f"git failed: {message}")
    return result


if (REPO_DIR / ".git").is_dir():
    _git(["-C", str(REPO_DIR), "fetch", "origin", "--tags", "--prune"])
else:
    _git(["clone", _auth_url(REPO_URL, token), str(REPO_DIR)])
    _git(["-C", str(REPO_DIR), "fetch", "origin", "--tags", "--prune"])

has_remote_branch = (
    _git(["-C", str(REPO_DIR), "rev-parse", "--verify", "--quiet", f"origin/{REF}"], check=False).returncode
    == 0
)
if has_remote_branch:
    checkout = _git(["-C", str(REPO_DIR), "checkout", "-B", REF, f"origin/{REF}"], check=False)
else:
    checkout = _git(["-C", str(REPO_DIR), "checkout", REF], check=False)
if checkout.returncode != 0:
    message = (checkout.stderr or checkout.stdout).replace(token, "***")
    raise RuntimeError(f"Could not check out REF={REF!r}: {message}")

subprocess.run(
    ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL],
    check=True,
)
del token
head = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print(f"repo ready at {head} (REF={REF})")

## 2. Install only missing Python packages, then check the GPU

Torch is already installed on Colab and is deliberately not touched.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "timm": "timm",
    "albumentations": "albumentations",
    "yaml": "pyyaml",
    "pytest": "pytest",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all required Python packages are already present")

import torch

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU. Evaluation is allowed on CPU but would be very slow; run on a GPU runtime.")

## 3. Prepare the datasets (raw -> processed, kept on Drive)

Reads `<DRIVE_DATA_ROOT>/raw/<folder>` and writes
`<DRIVE_DATA_ROOT>/processed/<name>/{data.npz,meta.json}` in the exact Kvasir
cache format. Datasets that are not on disk are skipped with a warning; a
prepared dataset at the same size is skipped. Rebuild with
`prepare_datasets.py --force` (or delete the npz) to change the size.

In [ ]:
import subprocess
import sys
from pathlib import Path

raw_root = Path(DRIVE_DATA_ROOT) / "raw"
if RUN_PREP:
    if not raw_root.is_dir():
        raise FileNotFoundError(
            f"No raw data at {raw_root}. Unpack each dataset to {raw_root}/<folder> "
            "(folder names are in configs/datasets.yaml), then rerun this cell."
        )
    Path(DRIVE_DATA_ROOT).mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "prepare_datasets.py"),
        "--size",
        str(PREP_SIZE),
        "--data-root",
        DRIVE_DATA_ROOT,
    ]
    print("prep command:", " ".join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, check=False)
    print(f"prepare exit code: {result.returncode}", flush=True)
    if result.returncode != 0:
        raise RuntimeError(f"prepare_datasets.py failed with exit code {result.returncode}")
else:
    print("RUN_PREP is False; reusing the processed tree on Drive.")

## 4. Copy `processed/` to `/content` and point `DATA_ROOT` at it

Evaluation reads every prepared dataset from `DATA_ROOT`, and Drive reads are
slow, so the npz tree is copied to `/content` once per session. The Kvasir
training split is evaluated through the run's config with its cache path
repointed at this local copy (see the next cell).

In [ ]:
import os
import shutil
from pathlib import Path

source = Path(DRIVE_DATA_ROOT) / "processed"
if not source.is_dir():
    raise FileNotFoundError(f"No processed tree at {source}; run the prep cell first.")

local_processed = Path(LOCAL_DATA_ROOT) / "processed"
if local_processed.exists():
    shutil.rmtree(local_processed)
shutil.copytree(source, local_processed)

os.environ["DATA_ROOT"] = LOCAL_DATA_ROOT
prepared = sorted(p.name for p in local_processed.iterdir() if (p / "data.npz").is_file())
print("DATA_ROOT:", os.environ["DATA_ROOT"])
print("prepared datasets:", prepared)

## 5. Choose the runs and datasets, and build session configs

A run's `config_resolved.yaml` records the data paths it trained with. Those
paths (`/content/...`) are gone in a fresh session, so this cell writes a
per-run session config that changes **only** the data paths: `data.split_dir`
points at the committed splits, and `data.cache_path` at the local prepared
Kvasir npz (byte-identical to the training cache). The architecture is untouched.
`src.evaluate --config` reads that file instead of the run's own config.

In [ ]:
import sys
from pathlib import Path

import yaml

sys.path.insert(0, str(REPO_DIR))
from src.data_registry import registry_names  # noqa: E402

runs_root = Path(RUNS_DIR)
if not runs_root.is_dir():
    raise FileNotFoundError(f"No runs folder at {runs_root}.")

found = []
for candidate in sorted(runs_root.iterdir()):
    if not candidate.is_dir():
        continue
    if any(bad in candidate.name for bad in EXCLUDE_SUBSTR):
        continue
    if not (candidate / "best.pt").is_file():
        continue
    if not (candidate / "config_resolved.yaml").is_file():
        continue
    found.append(candidate)

if ONLY_RUNS is not None:
    wanted = set(ONLY_RUNS)
    found = [path for path in found if path.name in wanted]
if not found:
    raise RuntimeError(f"No run folders to evaluate under {runs_root} (ONLY_RUNS={ONLY_RUNS}).")

# kvasir_seg first (the training dataset), then the prepared external datasets,
# both in registry order.
datasets = [name for name in registry_names() if (local_processed / name / "data.npz").is_file()]
if "kvasir_seg" not in datasets:
    raise RuntimeError("kvasir_seg is not prepared; the training dataset must be evaluated.")

session_dir = Path("/content/session_configs")
session_dir.mkdir(parents=True, exist_ok=True)
cache_path = str(local_processed / "kvasir_seg" / "data.npz")
session_configs = {}
for run_dir in found:
    cfg = yaml.safe_load((run_dir / "config_resolved.yaml").read_text(encoding="utf-8"))
    cfg.setdefault("data", {})
    cfg["data"]["root"] = str(REPO_DIR)
    cfg["data"]["split_dir"] = str(REPO_DIR / "data" / "splits")
    cfg["data"]["cache_path"] = cache_path
    path = session_dir / f"{run_dir.name}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    session_configs[run_dir] = path

print("runs:", [path.name for path in found])
print("datasets:", datasets)
print("session configs:", session_dir)

## 6. Evaluate each run on every prepared dataset

One row per `(run, split, dataset, tta)` is appended to `RESULTS_CSV` on Drive.
The Kvasir training dataset uses `EVAL_SPLIT` (`test` prints an explicit warning
and refuses to run on a leaking split); every external dataset is recorded as
`test`. `FORCE = True` makes a rerun replace its own row in place.

In [ ]:
import subprocess
import sys

for run_dir in found:
    cmd = [
        sys.executable,
        "-m",
        "src.evaluate",
        "--run_dir",
        str(run_dir),
        "--config",
        str(session_configs[run_dir]),
        "--split",
        EVAL_SPLIT,
        "--checkpoint",
        "best",
        "--results_csv",
        str(RESULTS_CSV),
    ]
    for name in datasets:
        cmd += ["--dataset", name]
    if TTA_FLIP:
        cmd.append("--tta-flip")
    if FORCE:
        cmd.append("--force")
    print(f"\n=== {run_dir.name} ===", flush=True)
    print("evaluate command:", " ".join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, check=False)
    print(f"evaluate exit code: {result.returncode}", flush=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"src.evaluate failed for {run_dir.name} (exit {result.returncode}). "
            "Check the checkpoint, the prepared datasets and the results CSV."
        )

## 7. Aggregate into one table

Rows = configs, columns = datasets, cells = `mean +/- std` over seeds (`n/a` for
a single seed or an unevaluated cell). Writes `aggregate_results.md` and
`aggregate_results.csv` into `RUNS_DIR`.

In [ ]:
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable,
    str(REPO_DIR / "scripts" / "aggregate_results.py"),
    "--runs-dir",
    RUNS_DIR,
    "--out-dir",
    RUNS_DIR,
    "--split",
    EVAL_SPLIT,
    "--tta",
    "flip" if TTA_FLIP else "",
]
print("aggregate command:", " ".join(cmd), flush=True)
result = subprocess.run(cmd, cwd=REPO_DIR, text=True, check=False)
print(f"aggregate exit code: {result.returncode}", flush=True)
if result.returncode != 0:
    raise RuntimeError(f"aggregate_results.py failed with exit code {result.returncode}")
print((Path(RUNS_DIR) / "aggregate_results.md").read_text(encoding="utf-8"))

## 8. Where everything lives

On Drive (`RUNS_DIR`, e.g. `/content/drive/MyDrive/polyp-runs`):

- `<run name>/best.pt` - the checkpoint selected by Kvasir val Dice (used here).
- `<run name>/last.pt` - the resumable checkpoint.
- `<run name>/config_resolved.yaml`, `run_info.json`, `metrics.csv`.
- `<run name>/eval_<split>_<dataset>[_ttaflip].json` and
  `per_image_<split>_<dataset>[_ttaflip].csv`.
- `results.csv` - one row per `(run, split, dataset, tta)`.
- `aggregate_results.md` / `aggregate_results.csv`.

On Drive (`DRIVE_DATA_ROOT`): `raw/<folder>` (read-only source) and
`processed/<name>/{data.npz,meta.json}`.

Ephemeral (rebuilt each session): `/content/polyp`, `/content/polyp-data`,
`/content/session_configs`.